In [2]:
# Vector Data impimentation
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [3]:
loader = PyPDFLoader("../test_documents/attention.pdf")

In [4]:
document = loader.load()

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
documents =  text_splitter.split_documents(document)
vectors = FAISS.from_documents(documents, embeddings)

In [6]:
retriver = vectors.as_retriever()

In [7]:
from langchain_core.tools import tool

@tool
def database_search(query: str) -> str:
    """Search for information about the research paper.
    You should use this tool to get the information about the relevant research paper."""
    docs = retriver.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [8]:
parse_agent_Tools = [database_search]

# Agent Implimentation 

# Document Parse Agent

In [9]:
## Agents
from langchain.agents import create_agent
from langchain.tools import tool


from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
load_dotenv()


True

In [10]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
parse_llm_with_tools = llm.bind_tools(parse_agent_Tools)

In [11]:
parse_agent_prompt = """You are the Research Document Parsing Agent. 
First Use the given retriver_tool Tool to get research paper context and  You Read the given  research paper  Fully and divide them into Abstract, Method, Math, Experiments, Results.
Rules:
- Do NOT wait for user input or paper name
- Go directly to the vector database and extract
- Do NOT ask questions
- Use the document retrieval tool to read the paper
- Be concise but complete
"""

In [12]:
parse_agent = create_agent(
    model=parse_llm_with_tools,  # or any other supported model identifier string
    tools=parse_agent_Tools,
    system_prompt=parse_agent_prompt
)

In [13]:
inputs = {"messages": [{"role": "user", "content": "Analyze the given research paper"}]}
result_parse_agent=parse_agent.invoke(inputs)
print(result_parse_agent["messages"][-1].content)

**Attention Is All You Need** – Ashish Vaswani et al. (2017)

---

## 1. Abstract  
The paper introduces the **Transformer**, a sequence‑to‑sequence architecture that relies **exclusively on attention mechanisms** and eliminates recurrence and convolution. Experiments on two machine‑translation benchmarks (WMT 2014 English‑German and WMT 2014 English‑French) show that the Transformer outperforms state‑of‑the‑art recurrent models while being far more parallelizable and requiring significantly fewer training steps.

---

## 2. Method (Model Architecture)

| Component | Description |
|-----------|-------------|
| **Encoder** | Stack of *N* identical layers (default *N* = 6). Each layer has: <br>• **Multi‑head self‑attention** (queries, keys, values all from the previous encoder layer). <br>• **Position‑wise feed‑forward network** (two linear layers with ReLU). |
| **Decoder** | Stack of *N* identical layers (default *N* = 6). Each layer has: <br>• **Masked multi‑head self‑attention** (pre

In [14]:
result_parse_agent

{'messages': [HumanMessage(content='Analyze the given research paper', additional_kwargs={}, response_metadata={}, id='477d97bc-97df-4159-b751-5572afa492c9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the database_search tool to retrieve the research paper. But we don\'t have a query. The instruction says "Do NOT wait for user input or paper name". So we need to decide a query. Maybe we can search for a generic research paper? But we need to analyze a given research paper. The user didn\'t provide a title. The instruction says "First Use the given retriver_tool Tool to get research paper context and You Read the given research paper Fully and divide them into Abstract, Method, Math, Experiments, Results." So we need to fetch a paper. But which one? The instruction says "Do NOT wait for user input or paper name". So we must choose a paper. Maybe we can search for a recent paper on a topic. But we need to produce a structured summary. We can pick a w

# Teaching Agent

In [15]:
teaching_agent_prompt = """You are the Teaching Assistant Agent. 
Read the given context carefully and explain the key concepts, Architecture ideas , algorithm intuition using 
simple language, Analogies and step by step reasoning so even a complete beginner could understand.Use the tools only if you want.
Compress the result into bullet points.
Keep only essential facts, findings, and conclusions.
"""

In [16]:
wiki_api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki_tool = WikipediaQueryRun(api_wrapper= wiki_api_wrapper)

In [17]:
llm = ChatGroq(
    model="openai/gpt-oss-safeguard-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
teach_tools = [wiki_tool]
teach_llm=llm.bind_tools(teach_tools)

In [18]:
teach_agent = create_agent(
    model=teach_llm,  # or any other supported model identifier string
    tools=teach_tools,
    system_prompt=teaching_agent_prompt
)

In [19]:
teach_agent_input = {"messages": [{"role": "user", "content": "Teach me this research paper context"+ "context - "+str(result_parse_agent["messages"][-1].content)}]}
teach_agent_results=teach_agent.invoke(teach_agent_input)
print(teach_agent_results["messages"][-1].content)

**Attention Is All You Need – “The Transformer” (2017)**  
*What it does:* A new way to turn one sentence into another (e.g., English → German) that uses **only attention** instead of the older “recurrent” or “convolution” tricks.

---

### 1. Why it matters
- **Fast & parallel** – all words can be processed at once, not one after another.  
- **Better quality** – beats the best old models on two big translation tests.  
- **Simplicity** – the whole network is built from the same two building blocks.

---

### 2. Core building blocks

| Block | What it does | Analogy |
|-------|--------------|---------|
| **Token embeddings** | Turns each word into a 512‑dim vector | Like giving each word a “color code” that the model can understand |
| **Positional encoding** | Adds a “time stamp” to each word so the model knows order | Like putting a number on each word’s position (1st, 2nd, …) using sine/cosine waves |
| **Multi‑head self‑attention** | Lets every word look at every other word and de

In [20]:
teach_agent_results

{'messages': [HumanMessage(content='Teach me this research paper contextcontext - **Attention Is All You Need** – Ashish Vaswani\u202fet\u202fal. (2017)\n\n---\n\n## 1. Abstract  \nThe paper introduces the **Transformer**, a sequence‑to‑sequence architecture that relies **exclusively on attention mechanisms** and eliminates recurrence and convolution. Experiments on two machine‑translation benchmarks (WMT\u202f2014 English‑German and WMT\u202f2014 English‑French) show that the Transformer outperforms state‑of‑the‑art recurrent models while being far more parallelizable and requiring significantly fewer training steps.\n\n---\n\n## 2. Method (Model Architecture)\n\n| Component | Description |\n|-----------|-------------|\n| **Encoder** | Stack of *N* identical layers (default *N*\u202f=\u202f6). Each layer has: <br>• **Multi‑head self‑attention** (queries, keys, values all from the previous encoder layer). <br>• **Position‑wise feed‑forward network** (two linear layers with ReLU). |\n| 

# Mathematical Reasoning Agent

In [21]:
Math_agent_prompt = """You are the Mathematical Reasoning Agent. 
Read the given research paper context carefully, extract all the mathematical formulas and explain , analyze equations, Loss functions, 
optimization methods.
Explain what each equation represents, why it needed and assumptions made so even a complete beginner could understand.
present the results in a human-readable format. 
Compress the result into bullet points.
Keep only essential facts, findings, and conclusions.
Use the provided tools ONLY if you need.

"""

In [22]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
math_tools = [wiki_tool]
math_llm=llm.bind_tools(math_tools)

In [23]:
math_agent = create_agent(
    model=math_llm,  # or any other supported model identifier string
    tools=math_tools,
    system_prompt=Math_agent_prompt
)

In [24]:
math_agent_input = {"messages": [{"role": "user", "content": "explain all the mathematical terms in this research paper"+ "context - "+str(result_parse_agent["messages"][-1].content)}]}
math_agent_results=teach_agent.invoke(math_agent_input)
print(math_agent_results["messages"][-1].content)

**Key math ideas in “Attention Is All You Need” – explained simply**

| Concept | What it is | Why it matters | Quick analogy |
|---------|------------|----------------|----------------|
| **Token embeddings** | Each word is turned into a vector of size *d_model* (512 in the paper). | Gives the model a numeric “picture” of each word. | Like giving every word a unique ID card that also carries extra “features” (e.g., part‑of‑speech). |
| **Queries (Q), Keys (K), Values (V)** | Three matrices derived from the same input (or from encoder/decoder). | They let the model decide *how much* one word should pay attention to another. | Think of Q as a question, K as a list of possible answers, V as the actual answers. |
| **Dot‑product attention** | `QKᵀ` computes a score for every pair of words. | Measures similarity: higher score → more attention. | Like comparing two people’s interests; if they match, they talk more. |
| **Scaling by √d_k** | Divide the dot‑product scores by the square root o

In [25]:
math_agent_results

{'messages': [HumanMessage(content='explain all the mathematical terms in this research papercontext - **Attention Is All You Need** – Ashish Vaswani\u202fet\u202fal. (2017)\n\n---\n\n## 1. Abstract  \nThe paper introduces the **Transformer**, a sequence‑to‑sequence architecture that relies **exclusively on attention mechanisms** and eliminates recurrence and convolution. Experiments on two machine‑translation benchmarks (WMT\u202f2014 English‑German and WMT\u202f2014 English‑French) show that the Transformer outperforms state‑of‑the‑art recurrent models while being far more parallelizable and requiring significantly fewer training steps.\n\n---\n\n## 2. Method (Model Architecture)\n\n| Component | Description |\n|-----------|-------------|\n| **Encoder** | Stack of *N* identical layers (default *N*\u202f=\u202f6). Each layer has: <br>• **Multi‑head self‑attention** (queries, keys, values all from the previous encoder layer). <br>• **Position‑wise feed‑forward network** (two linear lay

# Experimental Analysis Agent

In [29]:
experimental_analysis_agent_prompt = """You are the Experimental Analysis Agent and Empirical reviewer. 
Read the given context carefully and explain / Reviews Datasets used, Experimental setup, Baselines, Evaluation metrics.
Determines whether experiments support claims, If comparisons are fair. Conpare them with other related research papers.
Compress the result into bullet points.
Keep only essential facts, findings, and conclusions.
Use the given tools only if needed. Give me exacly a final answer don't think too much.
"""

In [30]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv_tool = ArxivQueryRun(api_wrapper= arxiv_wrapper)

In [31]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
experimental_analysis_agent_tools = [arxiv_tool,wiki_tool]
experimental_analysis_agent_llm=llm.bind_tools(experimental_analysis_agent_tools)

In [32]:
experimental_analysis_agent = create_agent(
    model=experimental_analysis_agent_llm,  # or any other supported model identifier string
    tools=experimental_analysis_agent_tools,
    system_prompt=experimental_analysis_agent_prompt
)

In [35]:
experimental_analysis_agent_input = {"messages": [{"role": "user", "content": "experiment analysis on the given research paper"+ "context - "+str(result_parse_agent["messages"][-1].content)}]}
experimental_analysis_agent_results=experimental_analysis_agent.invoke(experimental_analysis_agent_input)
print(experimental_analysis_agent_results["messages"][-1].content)

**Paper:** *Attention Is All You Need* (Vaswani et al., 2017) – introduces the Transformer architecture.  

---

### Datasets Used
- **WMT 2014 English→German (En‑De)**
  - ~4.5 M sentence pairs (≈ 4.5 B tokens after BPE).
- **WMT 2014 English→French (En‑Fr)**
  - ~36 M sentence pairs (≈ 36 B tokens after BPE).  
- Both are standard NMT benchmarks; test sets are the official newstest‑2014/2015 splits, enabling direct comparison with prior work.

### Experimental Setup
| Aspect | Details |
|--------|---------|
| **Model size** – *Base*: 6 encoder + 6 decoder layers, 8 attention heads, $d_{\text{model}}=512$, $d_{\text{ff}}=2048$, ~65 M parameters. |
| **Training** – Adam (β₁=0.9, β₂=0.98, ε=10⁻⁹), batch size ≈ 25 k tokens, learning‑rate schedule: $lr = d_{\text{model}}^{-0.5}\cdot \text{min(step}^{-0.5},\; step\cdot warmup^{-1.5})$ with 4000‑step warm‑up. |
| **Regularization** – Dropout 0.1 (attention, residual, embeddings), label smoothing 0.1. |
| **Hardware** – Single NVIDIA Tesla V

In [36]:
experimental_analysis_agent_results

{'messages': [HumanMessage(content='experiment analysis on the given research papercontext - **Attention Is All You Need** – Ashish Vaswani\u202fet\u202fal. (2017)\n\n---\n\n## 1. Abstract  \nThe paper introduces the **Transformer**, a sequence‑to‑sequence architecture that relies **exclusively on attention mechanisms** and eliminates recurrence and convolution. Experiments on two machine‑translation benchmarks (WMT\u202f2014 English‑German and WMT\u202f2014 English‑French) show that the Transformer outperforms state‑of‑the‑art recurrent models while being far more parallelizable and requiring significantly fewer training steps.\n\n---\n\n## 2. Method (Model Architecture)\n\n| Component | Description |\n|-----------|-------------|\n| **Encoder** | Stack of *N* identical layers (default *N*\u202f=\u202f6). Each layer has: <br>• **Multi‑head self‑attention** (queries, keys, values all from the previous encoder layer). <br>• **Position‑wise feed‑forward network** (two linear layers with R

# Testing Multi Agent Architecture

In [41]:
from langgraph_supervisor import create_supervisor
from langchain.chat_models import init_chat_model

In [42]:
llm_sup = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)

In [44]:
SUPERVISOR_PROMPT = """You are the research coordinator agent. 
    Read the given research paper context
    You can Breaks the paper into logical components such as problem, Method, Math (use the teach_math_tool for this),   
    Theory (use the teach_paper_tool for this), Experiments (use the experiment_analysis_tool for this), Results
    Assigns tasks to specialist agents, Controls analysis order, Provide agent inputs only with the relavant contenxt using the given tool.
    Give ONLY relevent CONTEXT TO EACH AGENT / TOOLS WITH INPUT.
    teach_math_tool - to explain math
    teach_paper_tool - to explain the paper content
    Decides when analysis is “complete” 
    Break down user requests into appropriate tool calls and coordinate the results. 
    When a request involves multiple actions, use multiple tools in sequence.
    finally you should give a full blown analysis descriptive report . Not Just A Summary.
    Full blown agent analysis
"""


In [45]:
supervisor = create_supervisor(
    model=llm_sup,
    agents=[teach_agent, math_agent,experimental_analysis_agent],
    prompt=(
        "You are a supervisor managing two agents:\n"
        "- a research agent. Assign research-related tasks to this agent\n"
        "- a math agent. Assign math-related tasks to this agent\n"
        "Assign work to one agent at a time, do not call agents in parallel.\n"
        "Do not do any work yourself."
    ),
    add_handoff_back_messages=True,
    output_mode="full_history",
).compile()

ValueError: Please specify a name when you create your agent, either via `create_react_agent(..., name=agent_name)` or via `graph.compile(name=name)`.

In [ ]:
from IPython.display import display, Image
display(Image(supervisor.get_graph().draw_mermaid_png()))

In [44]:
from langchain.tools import tool

In [ ]:
You are the Teaching Assistant Agent. 
Read the given context carefully and explain the key concepts, Architecture ideas , algorithm intuition using 
simple language, Analogies and step by step reasoning so even a complete beginner could understand.Use the tools only if you want.
Input: {input}
{agent_scratchpad}
<context>
{context}
</context>

In [46]:
@tool
async def teach_paper_tool(request: str, context: str) -> str:
    """Teach the given research paper context in a beginner friendly manner.

    Use this tool to explain the key concepts, Architecture ideas , algorithm intuition using 
    simple language, Analogies and step by step reasoning so even a complete beginner could understand.
    Compress the output content into bullet points.
    Keep only essential facts, findings, and conclusions.

    """
    combined_input = f"""
TASK:
{request}

RESEARCH PAPER CONTEXT:
{context}
"""
    result = await teaching_agent_executor.ainvoke({
        "input": combined_input
    })

    return result['output']




In [47]:
@tool
async def teach_math_tool(request: str, context: str) -> str:
    """Read the given research paper context carefully, extract all the mathematical formulas and explain , analyze equations, Loss functions, 
    optimization methods.

    Use this tool to Explain what each equation represents, why it needed and assumptions made so even a complete beginner could understand.
    present the results in a human-readable format.

    """
    combined_input = f"""
    TASK:
    {request}
    
    RESEARCH PAPER CONTEXT:
    {context}
    """
    result = await math_agent_executor.ainvoke({
        "input": combined_input
    })

    return result['output']

In [48]:
@tool
async def experiment_analysis_tool(request: str, context: str) -> str:
    """
    Use this tool to conduct the Experimental Analysis
    Read the given context carefully and explain / Reviews Datasets used, Experimental setup, Baselines, Evaluation metrics.
    Determines whether experiments support claims, If comparisons are fair. Conpare them with other related research papers.
    Only use a sufficient token amount so supervisor agent can handle it
    

    """
    combined_input = f"""
    TASK:
    {request}
    
    RESEARCH PAPER CONTEXT:
    {context}
    """
    result = await experimental_analysis_agent_executor.ainvoke({
        "input": combined_input
    })

    return result['output']

# Summarizer Agent

In [49]:
summarizer_agent_prompt = ChatPromptTemplate.from_template("""
    Compress the following content into bullet points.
    Keep only essential facts, findings, and conclusions.
    
    Content:
    {input}
    {agent_scratchpad}
    """) 

summary_llm = ChatGroq(
    model="openai/gpt-oss-safeguard-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params... 
)


#supervisor_tools = [teach_paper_tool, teach_math_tool, experiment_analysis_tool]
#supervisor_llm_with_tools =llm_supervisor.bind_tools(supervisor_tools)
summary_agent = create_tool_calling_agent(summary_llm,[wiki_tool], summarizer_agent_prompt)
summary_agent_executor = AgentExecutor(agent=summary_agent, tools=[wiki_tool], verbose=True,max_iterations=2,handle_parsing_errors=True)

C:\Anaconda3\Lib\site-packages\langchain_groq\chat_models.py:359: UserWarning: WARNING! reasoning_format is not default parameter.
                    reasoning_format was transferred to model_kwargs.
                    Please confirm that reasoning_format is what you intended.
  warnings.warn(


In [50]:
summarized_context = summary_agent_executor.invoke({"input":result_doc_agent["output"]})



> Entering new AgentExecutor chain...
**Attention Is All You Need – Key Points**

- **Core Idea**  
  - Introduces the *Transformer*, a seq‑to‑seq model that uses only attention mechanisms (no recurrence or convolution).  
  - Consists of an encoder and decoder, each with 6 identical layers of multi‑head self‑attention + position‑wise feed‑forward networks, plus residual + layer‑norm.

- **Architecture Highlights**  
  - **Encoder**: multi‑head self‑attention → feed‑forward.  
  - **Decoder**: masked self‑attention → encoder‑decoder attention → feed‑forward.  
  - **Multi‑head Attention**: split into 8 heads, scaled dot‑product, concatenate, linear projection.  
  - **Positional Encoding**: sinusoidal vectors added to embeddings (learned variants work similarly).  
  - **Output**: linear layer + softmax over vocabulary.

- **Mathematical Core**  
  - Scaled dot‑product: `Attention(Q,K,V)=softmax(QKᵀ/√dk)V`.  
  - Multi‑head: `Concat(head_i)WO` with `head_i=Attention(QWi^Q, Ki^K, Vi^V

In [51]:
print(summarized_context['output'])

**Attention Is All You Need – Key Points**

- **Core Idea**  
  - Introduces the *Transformer*, a seq‑to‑seq model that uses only attention mechanisms (no recurrence or convolution).  
  - Consists of an encoder and decoder, each with 6 identical layers of multi‑head self‑attention + position‑wise feed‑forward networks, plus residual + layer‑norm.

- **Architecture Highlights**  
  - **Encoder**: multi‑head self‑attention → feed‑forward.  
  - **Decoder**: masked self‑attention → encoder‑decoder attention → feed‑forward.  
  - **Multi‑head Attention**: split into 8 heads, scaled dot‑product, concatenate, linear projection.  
  - **Positional Encoding**: sinusoidal vectors added to embeddings (learned variants work similarly).  
  - **Output**: linear layer + softmax over vocabulary.

- **Mathematical Core**  
  - Scaled dot‑product: `Attention(Q,K,V)=softmax(QKᵀ/√dk)V`.  
  - Multi‑head: `Concat(head_i)WO` with `head_i=Attention(QWi^Q, Ki^K, Vi^V)`.  
  - Feed‑forward: `FFN(x)=max(0,xW

In [52]:
SUPERVISOR_PROMPT = ChatPromptTemplate.from_template(
    """You are the research coordinator agent. 
    Read the given research paper context
    You can Breaks the paper into logical components such as problem, Method, Math (use the teach_math_tool for this),   
    Theory (use the teach_paper_tool for this), Experiments (use the experiment_analysis_tool for this), Results
    Assigns tasks to specialist agents, Controls analysis order, Provide agent inputs only with the relavant contenxt using the given tool.
    Give ONLY relevent CONTEXT TO EACH AGENT / TOOLS WITH INPUT.
    teach_math_tool - to explain math
    teach_paper_tool - to explain the paper content
    Decides when analysis is “complete” 
    Break down user requests into appropriate tool calls and coordinate the results. 
    When a request involves multiple actions, use multiple tools in sequence.
    finally you should give a full blown analysis descriptive report . Not Just A Summary.
    Full blown agent analysis

    Input: {input}
    {agent_scratchpad}
    <context>
    {context}
    </context>"""
)

In [53]:

llm_supervisor = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)
supervisor_tools = [teach_paper_tool, teach_math_tool, experiment_analysis_tool]
supervisor_llm_with_tools =llm_supervisor.bind_tools(supervisor_tools)

C:\Anaconda3\Lib\site-packages\langchain_groq\chat_models.py:359: UserWarning: WARNING! reasoning_format is not default parameter.
                    reasoning_format was transferred to model_kwargs.
                    Please confirm that reasoning_format is what you intended.
  warnings.warn(


In [54]:
supervisor_agent = create_tool_calling_agent(supervisor_llm_with_tools,supervisor_tools, SUPERVISOR_PROMPT)
supervisor_agent_executor = AgentExecutor(agent=supervisor_agent, tools=supervisor_tools, verbose=True,max_iterations=10,handle_parsing_errors=True)

In [57]:
import asyncio

async def run_pipeline():

    result_doc_agent= await parse_agent_executor.ainvoke({'input':"analyze the  given research paper with the given database retriver tool"})


    summary_context= await summary_agent_executor.ainvoke({"input":result_doc_agent["output"]})
    
    result = await supervisor_agent_executor.ainvoke({
        "input": f"""
Analyze and explain the following research paper.
Use provided tools when appropriate.

"""
    ,'context':summary_context['output']})

    return result['output']


# Run
final_output = await run_pipeline()
print(final_output)




> Entering new AgentExecutor chain...

Invoking: `Database_Search` with `{'query': 'research paper'}`


around each of the sub-layers, followed by layer normalization. We also modify the self-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position ican depend only on the known outputs at positions less than i.
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.
3.2.1 Scaled Dot-Product Attention
We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of
queries and keys of dimens

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01k5t1nbyrf1baqpqqj4ncp2sq` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8414, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [56]:
print(final_output)

**Key Take‑aways from the “Attention Is All You Need” fragment**

- **Model architecture**
  - Purely attention‑based encoder–decoder stack.
  - Each layer has two sub‑layers: (1) self‑attention (masked in decoder) and (2) position‑wise feed‑forward network.
  - Residual connections + layer‑norm around every sub‑layer.

- **Scaled dot‑product attention**
  - Formula: `Attention(Q,K,V) = softmax((QKᵀ)/√d_k) V`.
  - Scaling by `1/√d_k` keeps dot‑product variance ≈ 1, preventing vanishing gradients in the softmax.

- **Multi‑head attention**
  - Parallel heads with independent linear projections `W^Q_i, W^K_i, W^V_i`.
  - Each head outputs a `d_v`‑dimensional vector; concatenated and linearly projected back to model dimension.
  - Enables simultaneous focus on different representation sub‑spaces.

- **Decoder masking**
  - Causal mask blocks a token from attending to future positions, ensuring autoregressive generation.
  - Output embeddings are offset by one position to enforce this cons